# Lecture 8: Matrix Multiplications Kernels (Part A)

## Kernel 1


In [ ]:
#%%writefile matmul-01.py
"""
load from matA -> coalesced
load from matB -> uncoalesced
store in matC -> uncoalesced
"""
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a, b: (a + b - 1) // b

@cuda.jit
def matmul(matA, matB, matC, m, n, k):
  out_row = cuda.threadIdx.x + cuda.blockDim.x * cuda.blockIdx.x
  out_col = cuda.threadIdx.y + cuda.blockDim.y * cuda.blockIdx.y
  if out_col >= n or out_row >= m:
    return
  dot_product = float32(0.0)
  for i in range(k):
    dot_product += matA[out_row, i] * matB[i, out_col]
  matC[out_row, out_col] = dot_product

m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m, k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k, n), dtype=cp.float32) * 3.0
matC = cp.zeros(shape=(m, n), dtype=cp.float32)

threads_per_block = (32, 32)
blocks_per_grid = (
    ceil_div(m, threads_per_block[0]),
    ceil_div(n, threads_per_block[1])
)
matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)

In [ ]:
!ncu --set full -f -o matmul-01.ncu-rep python3 matmul-01.py

==PROF== Connected to process 11176 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 33 passes
==PROF== Disconnected from process 11176
==PROF== Report: /content/matmul-01.ncu-rep


## Kernel 2

In [ ]:
%%writefile matmul-02.py
"""
load from matA -> coalesced
load from matB -> uncoalesced
store in matC -> coalesced
"""
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a, b: (a + b - 1) // b

@cuda.jit
def matmul(matA, matB, matC, m, n, k):
  out_col = cuda.threadIdx.x + cuda.blockDim.x * cuda.blockIdx.x
  out_row = cuda.threadIdx.y + cuda.blockDim.y * cuda.blockIdx.y
  if out_col >= n or out_row >= m:
    return
  dot_product = float32(0.0)
  for i in range(k):
    dot_product += matA[out_row, i] * matB[i, out_col]
  matC[out_row, out_col] = dot_product

m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m, k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k, n), dtype=cp.float32) * 3.0
matC = cp.zeros(shape=(m, n), dtype=cp.float32)

threads_per_block = (32, 32)
blocks_per_grid = (
    ceil_div(n, threads_per_block[0]), # x-direction
    ceil_div(m, threads_per_block[1])  # y-direction
)
matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)

Writing matmul-02.py


In [ ]:
!ncu --set full -f -o matmul-02.ncu-rep python3 matmul-02.py

==PROF== Connected to process 19009 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 33 passes
==PROF== Disconnected from process 19009
==PROF== Report: /content/matmul-02.ncu-rep


## Kernel 3

In [ ]:
%%writefile matmul-03.py
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a, b: (a + b - 1) // b
tile_dim = 32 # size of the tile

@cuda.jit
def matmul(matA, matB, matC, m, n, k):
  tileA = cuda.shared.array(shape=(tile_dim, tile_dim), dtype=cp.float32)
  tileB = cuda.shared.array(shape=(tile_dim, tile_dim), dtype=cp.float32)

  tx = cuda.threadIdx.x
  ty = cuda.threadIdx.y

  out_col = tx + cuda.blockIdx.x * cuda.blockDim.x
  out_row = ty + cuda.blockIdx.y * cuda.blockDim.y

  num_tiles = (k + tile_dim - 1) // tile_dim
  product = float32(0.0)
  for t in range(num_tiles):
    # phase 1:
    # threads within the block
    # will read elements from matA and matB (gmem)
    # and store them in tileA and tileB (smem)
    tile_offset = t * tile_dim
    b_row = tile_offset + ty
    a_col = tile_offset + tx

    # load values from matA (gmem) into tileA (smem)
    if out_row < m and a_col < k:
      tileA[ty, tx] = matA[out_row, a_col]
    else:
      tileA[ty, tx] = float32(0.0)

    if b_row < k and out_col < n:
      tileB[ty, tx] = matB[b_row, out_col]
    else:
      tileB[ty, tx] = float32(0.0)

    cuda.syncthreads()

    # phase 2:
    # threads will compute dot product
    # by reading elements from tileA and tileB (smem)
    for i in range(tile_dim):
      product += tileA[ty, i] * tileB[i, tx]

    cuda.syncthreads()

  if out_row < m and out_col < n:
    matC[out_row, out_col] = product

m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m, k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k, n), dtype=cp.float32) * 3.0
matC = cp.zeros(shape=(m, n), dtype=cp.float32)

threads_per_block = (tile_dim, tile_dim)
blocks_per_grid = (
    ceil_div(n, threads_per_block[0]),
    ceil_div(m, threads_per_block[1])
)
matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)

Writing matmul-03.py


In [ ]:
!ncu --set full -f -o matmul-03.ncu-rep python3 matmul-03.py

==PROF== Connected to process 34921 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 33 passes
==PROF== Disconnected from process 34921
==PROF== Report: /content/matmul-03.ncu-rep


## Kernel 4

In [ ]:
%%writefile matmul-04.py
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a, b: (a + b - 1) // b
BM = 32
BN = 32
BK = 32
TM = 8                # no. of output elements processed by a single thread
NT = (BM * BN) // TM  # no. of threads required within a block

@cuda.jit
def matmul(matA, matB, matC, m, n, k):
  tileA = cuda.shared.array(shape=(BM, BK), dtype=cp.float32)
  tileB = cuda.shared.array(shape=(BK, BN), dtype=cp.float32)

  tid = cuda.threadIdx.x
  col = tid % BN
  row = (tid // BN) * TM

  block_row = cuda.blockIdx.y * BM
  block_col = cuda.blockIdx.x * BN

  # register allocation and initialization
  product = cuda.local.array(TM, dtype=cp.float32)
  for t in range(TM):
    product[t] = float32(0.0)


  for k0 in range(0, k, BK):
    # phase 1:
    # threads will load values from matA and matB
    # into tileA and tileB
    # gmem -> smem

    # phase 1: load values from matA into tileA
    for i in range(tid, BM * BK, NT):
      # convert 'linear' tid
      # to 2D coordinates (r, c)
      r = i // BK
      c = i % BK
      global_mem_row = block_row + r
      global_mem_col = k0 + c
      if global_mem_col < k and global_mem_row < m:
        tileA[r, c] = matA[global_mem_row, global_mem_col]
      else:
        tileA[r, c] = float32(0.0)
    # phase 1: load values from matB into tileB
    for i in range(tid, BK * BN, NT):
      r = i // BN
      c = i % BN
      global_mem_row = k0 + r
      global_mem_col = block_col + c
      if global_mem_row < k and global_mem_col < n:
        tileB[r, c] = matB[global_mem_row, global_mem_col]
      else:
        tileB[r, c] = float32(0.0)

    cuda.syncthreads()

    # phase 2
    for kk in range(BK):
      b = tileB[kk, col]
      for t in range(TM):
        product[t] += tileA[row + t, kk] * b

    cuda.syncthreads()

  global_mem_col = block_col + col
  for t in range(TM):
    global_mem_row = block_row + row + t
    if global_mem_row < m and global_mem_col < n:
      matC[global_mem_row, global_mem_col] = product[t]

m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m, k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k, n), dtype=cp.float32) * 3.0
matC = cp.zeros(shape=(m, n), dtype=cp.float32)

threads_per_block = (
    NT, 1
)
blocks_per_grid = (
    ceil_div(n, BN),
    ceil_div(m, BM)
)
matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)

Writing matmul-04.py


In [ ]:
!ncu --set full -f -o matmul-04.ncu-rep python3 matmul-04.py

==PROF== Connected to process 55072 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 33 passes
==PROF== Disconnected from process 55072
==PROF== Report: /content/matmul-04.ncu-rep
